In [5]:
from pathlib import Path
# 
import numpy as np
import os
from utils.benchmarks import BENCHMARKS_TO_PATHS
if "IREWR_WITH_MODIN" in os.environ and os.environ["IREWR_WITH_MODIN"] == "True":
    import os
    os.environ["MODIN_ENGINE"] = "ray"
    import ray
    ray.init(num_cpus=int(os.environ['MODIN_CPUS']), runtime_env={'env_vars': {'__MODIN_AUTOIMPORT_PANDAS__': '1'}})
    import modin.pandas as pd
else:
    import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import time

In [6]:
start_time = time.time()

In [7]:
passmark = 40

In [8]:
### cell 0 ###

benchmark_name = "student-performance-in-exams"
df = pd.read_csv(Path(BENCHMARKS_TO_PATHS[benchmark_name]).parent / "input" / "StudentsPerformance.csv")
factor = 100000
df = pd.concat([df]*factor)
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 100000000 entries, 0 to 999
Data columns (total 8 columns):
 #   Column                       Dtype 
---  ------                       ----- 
 0   gender                       object
 1   race/ethnicity               object
 2   parental level of education  object
 3   lunch                        object
 4   test preparation course      object
 5   math score                   int64 
 6   reading score                int64 
 7   writing score                int64 
dtypes: int64(3), object(5)
memory usage: 6.7+ GB


In [9]:
### cell 1 ###

df.isna().sum()

gender                         0
race/ethnicity                 0
parental level of education    0
lunch                          0
test preparation course        0
math score                     0
reading score                  0
writing score                  0
dtype: int64

In [10]:
### cell 2 ###

df.head()

,gender,race/ethnicity,parental level of education,lunch,test preparation course,math score,reading score,writing score
0,female,group B,bachelor's degree,standard,none,72,72,74
1,female,group C,some college,standard,completed,69,90,88
2,female,group B,master's degree,standard,none,90,95,93
3,male,group A,associate's degree,free/reduced,none,47,57,44
4,male,group C,some college,standard,none,76,78,75


In [11]:
### cell 3 ###

df.describe()

,math score,reading score,writing score
count,1.000000e+08,1.000000e+08,1.000000e+08
mean,6.608900e+01,6.916900e+01,6.805400e+01
std,1.515550e+01,1.459289e+01,1.518806e+01
min,0.000000e+00,1.700000e+01,1.000000e+01
25%,5.700000e+01,5.900000e+01,5.775000e+01
50%,6.600000e+01,7.000000e+01,6.900000e+01
75%,7.700000e+01,7.900000e+01,7.900000e+01
max,1.000000e+02,1.000000e+02,1.000000e+02


In [12]:
### cell 4 ###

df.isnull().sum()

gender                         0
race/ethnicity                 0
parental level of education    0
lunch                          0
test preparation course        0
math score                     0
reading score                  0
writing score                  0
dtype: int64

In [13]:
### cell 5 ###

df['math score'] = pd.to_numeric(df['math score'], errors='coerce')
df['Math_PassStatus'] = np.where(df['math score']<passmark, 'F', 'P')
df.Math_PassStatus.value_counts()

Math_PassStatus
P    96000000
F     4000000
Name: count, dtype: int64

In [14]:
### cell 6 ###

df['reading score'] = pd.to_numeric(df['reading score'], errors='coerce')
df['Reading_PassStatus'] = np.where(df['reading score']<passmark, 'F', 'P')
df.Reading_PassStatus.value_counts()

Reading_PassStatus
P    97400000
F     2600000
Name: count, dtype: int64

In [15]:
### cell 7 ###

df['writing score'] = pd.to_numeric(df['writing score'], errors='coerce')
df['Writing_PassStatus'] = np.where(df['writing score']<passmark, 'F', 'P')
df.Writing_PassStatus.value_counts()

Writing_PassStatus
P    96800000
F     3200000
Name: count, dtype: int64

In [16]:
### cell 8 ###

df['OverAll_PassStatus'] = df.apply(lambda x : 'F' if x['Math_PassStatus'] == 'F' or 
                                    x['Reading_PassStatus'] == 'F' or x['Writing_PassStatus'] == 'F' else 'P', axis =1)

df.OverAll_PassStatus.value_counts()

OverAll_PassStatus
P    94900000
F     5100000
Name: count, dtype: int64

In [17]:
### cell 9 ###

df['Total_Marks'] = df['math score']+df['reading score']+df['writing score']
df['Percentage'] = df['Total_Marks']/3

In [18]:
### cell 10 ###

def GetGrade(Percentage, OverAll_PassStatus):
    if ( OverAll_PassStatus == 'F'):
        return 'F'    
    if ( Percentage >= 80 ):
        return 'A'
    if ( Percentage >= 70):
        return 'B'
    if ( Percentage >= 60):
        return 'C'
    if ( Percentage >= 50):
        return 'D'
    if ( Percentage >= 40):
        return 'E'
    else: 
        return 'F'

df['Grade'] = df.apply(lambda x : GetGrade(x['Percentage'], x['OverAll_PassStatus']), axis=1)

df.Grade.value_counts()

Grade
B    26100000
C    25600000
A    19800000
D    17800000
E     5600000
F     5100000
Name: count, dtype: int64

In [19]:
end_time = time.time()
print(end_time - start_time)

1540.4460883140564
